# TO DO
Most of the pages have already been scraped and parsed into .csv files.

Open these, clean them and interpret the data in any meaningful way.

1. Ensure the datatype is correct.
2. Ensure the null, NaN, default is correct.
3. Collapse redundant info : Genres = Genre

In [1]:
import pandas as pd
import numpy as np
import os
import re

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing Submissions Data

In [2]:
FOLDER = 'Statistics/Submissions'

FILENAMES = [f'{FOLDER}/{file}' for file in os.listdir(FOLDER)[1:]]
FILENAMES

['Statistics/Submissions/K-On_Movie_sample.csv',
 'Statistics/Submissions/Shinseiki Evangelion.csv']

In [3]:
df_Submission_Raw = pd.read_csv(FILENAMES[1], index_col=1).drop(columns='Unnamed: 0')
df_Submission_Raw

,Score,Status,Eps Seen,Activity
Member,,,,
mengor034,-,Completed,26 / 26,57 minutes ago
IlBravu,-,Watching,18 / 26,60 minutes ago
happieness_9,-,Watching,3 / 26,1 hour ago
Elijahuh,10,Completed,26 / 26,1 hour ago
cxrgx,-,On-Hold,7 / 26,1 hour ago
...,...,...,...,...
greentriangle,9,Completed,26 / 26,"Jan 7, 11:15 PM"
szuszel,-,Plan to Watch,NaN,"Jan 7, 11:11 PM"
itsjustabyss,10,Completed,26 / 26,"Jan 7, 11:03 PM"


# Begin Cleaning

In [4]:
df_Cleaning = df_Submission_Raw.copy()

In [5]:
df_Cleaning.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7500 entries, mengor034 to LP98
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Score     7500 non-null   object
 1   Status    7499 non-null   object
 2   Eps Seen  6147 non-null   object
 3   Activity  7500 non-null   object
dtypes: object(4)
memory usage: 293.0+ KB


## Convert the data types
There's currently strings and a lack of NaN

### Score
> replace - into NaN

In [6]:
df_Cleaning.Score.unique()

array(['-', '10', '7', '6', '9', '8', '5', '4', '3', '2', '1'],
      dtype=object)

In [7]:
df_Cleaning.Score = df_Cleaning.Score.replace('-', np.nan).astype(float)
df_Cleaning.Score.value_counts()

Score
10.0    1200
9.0      953
8.0      870
7.0      500
6.0      189
5.0       86
4.0       39
3.0       14
1.0       13
2.0       11
Name: count, dtype: int64

### Eps Seen
> replace - to NaN, and extract only the episodes seen

In [8]:
df_Cleaning['Eps Seen'].unique()

array(['26 / 26', '18 / 26', '3 / 26', '7 / 26', '17 / 26', '23 / 26',
       nan, '21 / 26', '1 / 26', '5 / 26', '24 / 26', '14 / 26', '4 / 26',
       '15 / 26', '6 / 26', '2 / 26', '- / 26', '25 / 26', '8 / 26',
       '12 / 26', '16 / 26', '20 / 26', '10 / 26', '11 / 26', '13 / 26',
       '22 / 26', '19 / 26', '9 / 26'], dtype=object)

In [9]:
df_Cleaning['Eps Seen'] = df_Cleaning['Eps Seen'].replace("\\s\\/.+\\d$", '', regex=True)
df_Cleaning['Eps Seen'] = df_Cleaning['Eps Seen'].replace('-', np.nan).astype(float)
df_Cleaning['Eps Seen'].value_counts()

Eps Seen
26.0    4560
1.0      127
3.0      105
2.0       93
12.0      73
4.0       67
6.0       61
7.0       61
5.0       59
10.0      59
11.0      54
8.0       50
13.0      50
9.0       40
24.0      36
16.0      32
20.0      30
21.0      29
14.0      29
15.0      27
17.0      24
18.0      22
22.0      22
25.0      19
23.0      18
19.0      18
Name: count, dtype: int64

### Activity

In [10]:
def ActivityExtract(x):
    res = re.findall(r'\b(?:second|minute|hour|yesterday)', x, re.IGNORECASE)

    if len(res) > 0:
        return res[0].lower()
    else:
        return 'days'

In [11]:
df_Cleaning.Activity = df_Cleaning.Activity.apply(lambda x : ActivityExtract(x))
df_Cleaning.Activity.value_counts()

Activity
days         6844
yesterday     425
hour          229
minute          2
Name: count, dtype: int64

In [12]:
df_Cleaning

,Score,Status,Eps Seen,Activity
Member,,,,
mengor034,NaN,Completed,26.0,minute
IlBravu,NaN,Watching,18.0,minute
happieness_9,NaN,Watching,3.0,hour
Elijahuh,10.0,Completed,26.0,hour
cxrgx,NaN,On-Hold,7.0,hour
...,...,...,...,...
greentriangle,9.0,Completed,26.0,days
szuszel,NaN,Plan to Watch,NaN,days
itsjustabyss,10.0,Completed,26.0,days


# Dropping the Failed Scrapes

In [13]:
df_Cleaning.isna().sum()

Score       3625
Status         1
Eps Seen    1735
Activity       0
dtype: int64

In [15]:
assert False

AssertionError: 

In [16]:
df_Cleaning[df_Cleaning['Status'].isna()]

,Score,Status,Eps Seen,Activity
Member,,,,
Dhanush_urs,NaN,NaN,NaN,hour


In [17]:
df_Cleaning = df_Cleaning.drop(df_Cleaning[df_Cleaning['Status'].isna()].index)

# One Hot Encoding
> We can now encode our categorical values.

In [ ]:
df_OneHot = pd.get_dummies(df_Cleaning, columns=['Status', 'Activity'])
df_OneHot

In [ ]:
df_OneHot

In [ ]:
df_OneHot.info()

# Visualize

In [ ]:
df_Visualize = df_OneHot.copy()
df_Visualize['Viewership'] = df_Cleaning.Status
df_Visualize.groupby('Viewership').mean()

In [ ]:
df = df_Cleaning[['Status', 'Eps Seen', 'Score']]

In [ ]:
fig = px.histogram(df.Status,labels={'value':'Status'}, title='Viewing Status')

fig.show()

In [ ]:
fig = px.histogram(df['Eps Seen'], labels={'value':'Episode'}, title='Total Episodes Seen')

fig.show()

In [ ]:
df_Scatter = df.groupby(['Score', 'Eps Seen']).value_counts()
df_Scatter

In [ ]:
#fig = px.scatter(df_Scatter.reset_index(), y='Score', x='Eps Seen', color='Status', size='count')
fig = px.scatter(df_Scatter.reset_index(), y='Score', x='Eps Seen', color='Status')

fig.show()